In [3]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import seaborn as sns
import math

np.random.seed(11)
n = 100

In [4]:
theta_true = 42
sample = stats.uniform(loc=theta_true, scale=theta_true).rvs(size=n)
print(f"Выборка: {sample}")

Выборка: [49.57132693 42.81796014 61.45517811 72.44722503 59.64855139 62.38793812
 42.53679421 62.46960751 81.5558794  77.73339375 72.65850775 46.56691502
 79.54397515 78.00047838 48.93363794 68.55802858 42.86031174 46.90296529
 55.28742709 48.63231688 73.8771427  76.36756503 56.47422862 55.38954947
 46.68977174 45.52603202 71.9344893  67.18082264 44.33829454 62.15148583
 58.87041219 77.6151179  72.14966553 67.28669015 65.2001205  81.86230071
 83.44027988 56.19827009 52.07473652 75.45030167 44.67483009 57.31385701
 44.94095771 55.4134437  44.95606899 54.19107402 75.18424718 80.02681347
 75.29009817 65.59638583 67.87277217 57.18230877 49.09032654 60.32211921
 72.77866413 44.6412802  42.87078511 74.36301879 54.5979844  71.44889955
 72.85604428 81.18199434 58.81379418 57.05439299 75.87580522 74.10862605
 69.40981908 76.06059611 68.97302407 82.21265022 56.0227257  73.00661016
 81.89299884 56.04329447 67.68854386 57.35814704 61.38469224 45.15008371
 42.81242122 73.90527764 61.01588491 68.15

In [5]:
sample_mean = np.mean(sample)
sample_max = np.max(sample)

theta_mm = 2/3 * sample_mean # ОММ
theta_mp = (n+1) / (2*n+1) * sample_max  # ОМП (смещенная)


In [11]:
beta = 0.95

a = ((1 + beta)/2)**(1/n)
b = ((1 - beta)/2)**(1/n)

accurate_left = sample_max / (1 + a)
accurate_right = sample_max / (1 + b)
accurate_length = accurate_right - accurate_left

print(f"Точный доверительный интервал для θ: {accurate_left:.4f} < θ < {accurate_right:.4f}")
print(f"Длина интервала: {accurate_length:.4f}")

Точный доверительный интервал для θ: 41.7254 < θ < 42.4896
Длина интервала: 0.7641


In [7]:
# Асимптотический доверительный интервал (ОММ)
sample_var = np.sum((sample - sample_mean)**2) / n #выборочная дисперсия
z_crit = stats.norm.ppf((1 + beta)/2)
std_error = 2/3 * np.sqrt(sample_var / n)

alpha2=  np.sum(sample**2)/n
alpha1= np.sum(sample)/n
asym_l = theta_mm - 2/3*np.sqrt(alpha2 - alpha1**2)/np.sqrt(n)*z_crit
asym_r = theta_mm + 2/3*np.sqrt(alpha2 - alpha1**2)/np.sqrt(n)*z_crit
asym_length = asym_r - asym_l

print(f"Асимптотический доверительный интервал для θ: {asym_l:.4f}, < θ < {asym_r:.4f}")
print(f"Длина интервала: {asym_length:.4f}")

Асимптотический доверительный интервал для θ: 40.6191, < θ < 43.7255
Длина интервала: 3.1063


In [13]:
# Непараметрический бутстраповский доверительный интервал для ОММ
N_bootstrap = 1000

def bootstrap_omm(sample_sample, B):
    bootstrap_stats = []
    n_sample = len(sample_sample)
    for _ in range(B):
        bootstrap_sample = np.random.choice(sample_sample, size=n_sample, replace=True)        
        bootstrap_stats.append(2/3 * np.mean(bootstrap_sample) - theta_mm)
    return sorted(np.array(bootstrap_stats))

bootstrap_arr = bootstrap_omm(sample, N_bootstrap)
quantile_idx_left = int((1 - beta)/2 * N_bootstrap - 1)
quantile_idx_right = int((1 + beta)/2 * N_bootstrap - 1)

bs_omm_left = theta_mm - bootstrap_arr[quantile_idx_right]
bs_omm_right = theta_mm - bootstrap_arr[quantile_idx_left]
bs_omm_length = bs_omm_right - bs_omm_left

print("Непараметрический бутстрап (ОММ):", bs_omm_left, "< θ <", bs_omm_right)
print("Длина интервала:", bs_omm_length)

Непараметрический бутстрап (ОММ): 40.67526759125295 < θ < 43.69644777098402
Длина интервала: 3.0211801797310756


In [12]:
# Непараметрический бутстраповский доверительный интервал для ОМП
def bootstrap_omp(sample_sample, B):
    bootstrap_stats = []
    n_sample = len(sample_sample)
    for _ in range(B):
        bootstrap_sample = np.random.choice(sample_sample, size=n_sample, replace=True)        
        bootstrap_stats.append((n_sample + 1) * np.max(bootstrap_sample) / (2 * n_sample + 1) - theta_mp)
    return sorted(np.array(bootstrap_stats))

bootstrap_arr = bootstrap_omp(sample, N_bootstrap)
quantile_idx_left = int((1 - beta)/2 * N_bootstrap)
quantile_idx_right = int((1 + beta)/2 * N_bootstrap)

bs_omp_left = theta_mp - bootstrap_arr[quantile_idx_right]
bs_omp_right = theta_mp - bootstrap_arr[quantile_idx_left]
bs_omp_length = bs_omp_right - bs_omp_left

print("Непараметрический бутстрап (ОМП):", bs_omp_left, "< θ <", bs_omp_right)
print("Длина интервала:", bs_omp_length)

Непараметрический бутстрап (ОМП): 41.927702822789186 < θ < 42.72061772895529
Длина интервала: 0.7929149061661036
